# Replicação Fiel do Artigo: VGG-16 nos 6 Cenários da Tabela 3
Este notebook executa o treinamento e a avaliação da CNN **VGG-16** treinada do zero (`weights=None`), exatamente como na metodologia do artigo científico:
> *"Preprocessing and Optimizer Impact on Image-Based Autoimmune Disease Diagnosis"* (RITA, 2025).

Ele cobre os **6 cenários da Tabela 3 do artigo** cruzando os fatores:
1. **Técnicas de Imagem:** Original (`none`), Expansão de Contraste (`contrast_expansion`) e Equalização de Histograma (`histogram_equalization`).
2. **Otimizadores:** `SGD` e `Adam`.

### 1. Verificar GPU e Atualizar Repositório
Certifique-se de que a GPU está ativa (**Ambiente de Execução > Alterar tipo de ambiente de execução > T4 GPU**).

In [ ]:
!nvidia-smi

In [ ]:
import os
if not os.path.exists("/content/Projeto-PDI"):
    !git clone -b feat/vgg-16 https://github.com/orleoncio/Projeto-PDI.git /content/Projeto-PDI

%cd /content/Projeto-PDI
!git checkout feat/vgg-16
!git pull origin feat/vgg-16

### 2. Instalar Dependências

In [ ]:
!pip install -q scikit-learn pandas pillow torchvision torch

### 3. Validar a Arquitetura da VGG-16

In [ ]:
%cd /content/Projeto-PDI
!python -u tests/test_vgg16.py

## 🧪 CENÁRIO 1: Imagem Original (Baseline)
Comparação entre SGD e Adam sem filtros na imagem de entrada.

#### 1.1 VGG-16 Original + SGD (Meta: 98.16% | Obtido anteriormente: ~89.67%)

In [ ]:
%cd /content/Projeto-PDI
!python -u main_vgg16.py --optimizer sgd --preprocessing none --epochs 30 --batch-size 32 --amp --num-workers 2

#### 1.2 VGG-16 Original + Adam (Meta: 19.96% | Obtido anteriormente: 21.73% - Colapso Confirmado)

In [ ]:
%cd /content/Projeto-PDI
!python -u main_vgg16.py --optimizer adam --preprocessing none --epochs 30 --batch-size 32 --amp --num-workers 2

## 🧪 CENÁRIO 2: Expansão de Contraste (Contrast Stretching)
O pré-processamento que, segundo o artigo, **resgata o Adam do colapso** e eleva a acurácia para mais de 86%.

#### 2.1 VGG-16 Contraste + SGD (Meta do Artigo: 97.76% acc | 97.58% F1)

In [ ]:
%cd /content/Projeto-PDI
!python -u main_vgg16.py --optimizer sgd --preprocessing contrast_expansion --epochs 30 --batch-size 32 --amp --num-workers 2

#### 2.2 VGG-16 Contraste + Adam (Meta do Artigo: 86.32% acc | 86.35% F1 — Fenômeno de Resgate!)

In [ ]:
%cd /content/Projeto-PDI
!python -u main_vgg16.py --optimizer adam --preprocessing contrast_expansion --epochs 30 --batch-size 32 --amp --num-workers 2

## 🧪 CENÁRIO 3: Equalização de Histograma (Histogram Equalization)
Redistribuição uniforme de intensidade de brilho em cada canal da imagem.

#### 3.1 VGG-16 Equalização + SGD (Meta do Artigo: 96.80% acc | 96.73% F1)

In [ ]:
%cd /content/Projeto-PDI
!python -u main_vgg16.py --optimizer sgd --preprocessing histogram_equalization --epochs 30 --batch-size 32 --amp --num-workers 2

#### 3.2 VGG-16 Equalização + Adam (Meta do Artigo: 20.81% acc | 5.74% F1 — Colapso Mantido)

In [ ]:
%cd /content/Projeto-PDI
!python -u main_vgg16.py --optimizer adam --preprocessing histogram_equalization --epochs 30 --batch-size 32 --amp --num-workers 2

## 📊 Confrontação Completa com a Tabela 3 do Artigo
Carrega os 6 modelos treinados e gera a tabela completa comparando as métricas obtidas contra os valores oficiais publicados.

In [ ]:
import torch
import pandas as pd
from data.hep2_dataset import create_dataloaders
from models.vgg16 import create_vgg16
from training.test import test_model

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Valores da Tabela 3 do artigo para a VGG-16 nos 6 cenários
paper_data = {
    ("Original", "SGD"): {"acc": 98.16, "f1": 98.05, "prec": 98.14, "rec": 97.98},
    ("Original", "Adam"): {"acc": 19.96, "f1": 5.55, "prec": 3.33, "rec": 16.67},
    ("Contraste", "SGD"): {"acc": 97.76, "f1": 97.58, "prec": 97.60, "rec": 97.56},
    ("Contraste", "Adam"): {"acc": 86.32, "f1": 86.35, "prec": 86.69, "rec": 86.69},
    ("Equalização", "SGD"): {"acc": 96.80, "f1": 96.73, "prec": 96.76, "rec": 96.71},
    ("Equalização", "Adam"): {"acc": 20.81, "f1": 5.74, "prec": 3.47, "rec": 16.67},
}

configs = [
    ("Original", "SGD", "vgg16_original_sgd.pth", "none"),
    ("Original", "Adam", "vgg16_original_adam.pth", "none"),
    ("Contraste", "SGD", "vgg16_contrast_expansion_sgd.pth", "contrast_expansion"),
    ("Contraste", "Adam", "vgg16_contrast_expansion_adam.pth", "contrast_expansion"),
    ("Equalização", "SGD", "vgg16_histogram_equalization_sgd.pth", "histogram_equalization"),
    ("Equalização", "Adam", "vgg16_histogram_equalization_adam.pth", "histogram_equalization"),
]

rows = []
for label_prep, label_opt, pth_file, prep_arg in configs:
    path = f"results/{pth_file}"
    try:
        use_norm = (label_opt == "SGD" or prep_arg == "contrast_expansion")
        _, test_loader = create_dataloaders(batch_size=32, preprocessing=prep_arg, normalize=use_norm)
        model = create_vgg16(num_classes=6, pretrained=False).to(device)
        model.load_state_dict(torch.load(path, map_location=device, weights_only=True))
        
        print(f"\n>>> Testando VGG-16 | Pré-processamento: {label_prep} | Otimizador: {label_opt} <<<")
        res = test_model(model, test_loader, device)
        
        p = paper_data[(label_prep, label_opt)]
        rows.append({
            "Pré-processamento": label_prep,
            "Otimizador": label_opt,
            "Acurácia Treino (%)": f"{res['accuracy']*100:.2f}%",
            "Acurácia Artigo (%)": f"{p['acc']:.2f}%",
            "Dif Acurácia (%)": f"{(res['accuracy']*100 - p['acc']):+.2f}%",
            "F1 Treino (%)": f"{res['f1']*100:.2f}%",
            "F1 Artigo (%)": f"{p['f1']:.2f}%",
            "Dif F1 (%)": f"{(res['f1']*100 - p['f1']):+.2f}%"
        })
    except FileNotFoundError:
        print(f"[-] Pesos {path} ainda não treinados/encontrados.")

if rows:
    df = pd.DataFrame(rows)
    print("\n===================== TABELA COMPARATIVA VGG-16 (TODOS OS CENÁRIOS) =====================")
    display(df)
